<a href="https://colab.research.google.com/github/yosbel-penate2/leak-algorithm-experiments/blob/main/bioinspired_leak_pdm_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# # Implementación del Algoritmo de la Gotera (Leak Algorithm)
# ## Reproducción de los experimentos del artículo
#
# Este notebook implementa el modelo bioinspirado de degradación por humedad,
# entrena la capa impermeabilizante y evalúa tres escenarios:
# 1. Funcionamiento normal (sin anomalías)
# 2. Degradación progresiva (120 anomalías)
# 3. Picos raros (25 anomalías)

import numpy as np
import matplotlib.pyplot as plt

# Fijar semilla para reproducibilidad
np.random.seed(42)

# Parámetros del modelo (según artículo)
n = 100                # número de celdas de la "tabla"
x_ref = 75.0           # temperatura nominal
beta = 2.0             # factor de escala para tamaño de gota
gamma = 0.1            # sensibilidad de la tangente hiperbólica
fe = 0.05              # coeficiente de evaporación
fd = 0.1               # coeficiente de degradación (constante si h>0)
fr = 0.02              # coeficiente de regeneración
h_sat = 100.0          # umbral de saturación de humedad
D_crit = 30.0          # umbral crítico de desgaste
train_steps = 2000     # pasos de entrenamiento (solo datos normales)
total_steps = 10000    # longitud total de la simulación

# ---------- Funciones auxiliares ----------
def cell_index(xi: float) -> int:
    """
    Calcula el índice de la celda destino (0‑based) a partir de la desviación xi.
    Fórmula del apéndice: i = min(n, max(1, floor(n/2*(1+tanh(γ*xi))) + 1))
    """
    val = (n / 2) * (1 + np.tanh(gamma * xi))
    idx = int(np.floor(val)) + 1      # 1‑based
    idx = min(n, max(1, idx))         # clamping
    return idx - 1                    # convertir a 0‑based

def redistribute(h: np.ndarray) -> np.ndarray:
    """
    Redistribuye el exceso de humedad (por encima de h_sat)
    equitativamente a los vecinos. Itera hasta que ninguna celda supere el umbral.
    """
    changed = True
    while changed:
        changed = False
        for i in range(n):
            if h[i] > h_sat:
                excess = h[i] - h_sat
                h[i] = h_sat
                # Distribuir a vecinos
                if i > 0 and i < n - 1:
                    h[i-1] += excess / 2.0
                    h[i+1] += excess / 2.0
                elif i == 0 and n > 1:
                    h[1] += excess
                elif i == n - 1 and n > 1:
                    h[n-2] += excess
                # Si n==1 la humedad excedente se pierde (no ocurre aquí)
                changed = True
    return h

def internal_update(h, v, b):
    """Ejecuta un paso de la dinámica interna (degradación, evaporación, regeneración, redistribución)."""
    # Degradación y evaporación
    for i in range(n):
        if h[i] > 0:
            v[i] = max(0.0, v[i] - fd)
            h[i] = max(0.0, h[i] - fe)
        # Regeneración (siempre que no esté al máximo)
        if v[i] < 100.0:
            v[i] = min(100.0, v[i] + fr)

    # Volcar buffer a humedad y luego redistribuir excesos
    for i in range(n):
        if b[i] > 0:
            h[i] += b[i]
            b[i] = 0.0

    h = redistribute(h)
    return h, v, b

# ---------- Simulación de un escenario ----------
def run_scenario(signal: np.ndarray):
    """
    Ejecuta el algoritmo completo sobre la señal 'signal' de longitud total_steps.
    Retorna: D_vals (serie de desgaste en fase de prueba), D_final, alarma (paso global),
             capa impermeabilizante, vector de vida final.
    """
    # Fase de entrenamiento (primeros train_steps, solo régimen normal)
    h = np.zeros(n)
    v = np.full(n, 100.0)
    b = np.zeros(n)
    for k in range(train_steps):
        x = signal[k]
        xi = x - x_ref
        if xi > 0:
            a = xi                     # g(xi) = max(0, xi)
            T = beta * a
            I = cell_index(xi)
            b[I] += T
        h, v, b = internal_update(h, v, b)

    # Construir capa impermeabilizante
    l_imperv = 100.0 - v   # degradación acumulada durante el entrenamiento

    # Reiniciar estado para la fase operativa
    h = np.zeros(n)
    v = np.full(n, 100.0)
    b = np.zeros(n)

    D_vals = np.zeros(total_steps - train_steps)
    alarm_step = None

    for k in range(train_steps, total_steps):
        x = signal[k]
        xi = x - x_ref
        if xi > 0:
            a = xi
            T = beta * a
            I = cell_index(xi)
            # Aplicar capa impermeabilizante
            T_eff = T * (1.0 - l_imperv[I] / 100.0)
            b[I] += T_eff

        h, v, b = internal_update(h, v, b)

        Dk = np.mean(100.0 - v)
        D_vals[k - train_steps] = Dk

        if alarm_step is None and Dk >= D_crit:
            alarm_step = k   # paso global (0‑based)

    D_final = np.mean(100.0 - v)
    return D_vals, D_final, alarm_step, l_imperv, v

# ---------- Generar señal base (ruido gaussiano alrededor del nominal) ----------
np.random.seed(123)   # semilla independiente para la señal
noise = np.random.normal(0, 0.5, total_steps)
signal_clean = x_ref + noise   # sin anomalías

# Escenario 1 – Normal
D1, Dfinal1, alarm1, l1, v1 = run_scenario(signal_clean.copy())

# Escenario 2 – Degradación progresiva (120 anomalías en fase de prueba)
np.random.seed(456)
anom_times2 = np.random.choice(np.arange(train_steps, total_steps), size=120, replace=False)
anom_amps2 = np.random.uniform(1, 5, 120)
signal2 = signal_clean.copy()
for t, amp in zip(anom_times2, anom_amps2):
    signal2[t] += amp

D2, Dfinal2, alarm2, l2, v2 = run_scenario(signal2)

# Escenario 3 – Picos raros (25 anomalías)
np.random.seed(789)
anom_times3 = np.random.choice(np.arange(train_steps, total_steps), size=25, replace=False)
anom_amps3 = np.random.uniform(1, 5, 25)
signal3 = signal_clean.copy()
for t, amp in zip(anom_times3, anom_amps3):
    signal3[t] += amp

D3, Dfinal3, alarm3, l3, v3 = run_scenario(signal3)

# ---------- Visualización ----------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Curvas de desgaste
ax1.plot(D1, label='Normal (0 anomalías)')
ax1.plot(D2, label='Degradación progresiva (120)')
ax1.plot(D3, label='Picos raros (25)')
ax1.axhline(D_crit, color='red', linestyle='--', label='Umbral crítico')
ax1.set_xlabel('Paso (fase de prueba)')
ax1.set_ylabel('Índice de desgaste $D_k$')
ax1.set_title('Evolución del desgaste')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Distribución final de vida en escenario 2
ax2.bar(range(1, n+1), v2, color='saddlebrown', edgecolor='black', linewidth=0.3)
ax2.set_xlabel('Celda')
ax2.set_ylabel('Vida residual $v_i$')
ax2.set_title('Distribución de vida al final (escenario 2)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ---------- Tabla de resultados ----------
print(f"{'Escenario':<30} {'Anomalías':<12} {'$D_{{final}}$':<10} {'Supera umbral':<15} {'Alarma (paso)':<15}")
print("-" * 85)
print(f"{'1. Normal (sin anomalías)':<30} {0:<12} {Dfinal1:<10.1f} {'No':<15} {'---':<15}")
print(f"{'2. Degradación progresiva':<30} {120:<12} {Dfinal2:<10.1f} {'Sí' if alarm2 else 'No':<15} {alarm2 if alarm2 else '---':<15}")
print(f"{'3. Picos raros (ruido)':<30} {25:<12} {Dfinal3:<10.1f} {'No':<15} {'---':<15}")